# 12. Sequential Models — per-customer LSTM + Transformer (FOC-175, phase F3)

The F3 question, fourth model family: **does per-customer transaction SEQUENCE
structure — "unusual for THIS customer" — add fraud signal beyond the
per-transaction arms, and does any of it survive the customer-disjoint axes
that collapsed every arm so far?**

Where the ladder stands (docs/FOC-174-report.md, nb7-nb11):

- **F0** — XGB baseline on the 10 transaction features: test PR-AUC 0.0532
  (chronological split).
- **F1** — base + client features (nb7 arm b): 0.2374 chronological — read
  there as customer-identity signal, not transferable demographics.
- **F2** — SCE 0.0340 / dictionary 0.1034 chronological (nb8); on the
  customer-grouped splits every arm collapsed to ~0.01, next to chance.
- **F3 so far** — gbdt-ensemble (nb9) 0.0145 and tabnet (nb10) 0.0138 on the
  PRIMARY random-grouped axis — chance-level; only the chronological axis
  (24 test positives, identity signal in play) separates arms from chance.
  nb11's TimesFM forecast-residual features did not lift xgb-client.
- **F3 primary axis** — random-grouped (FOC-175): customers assigned to
  train/test by a seeded random draw — customer-disjoint, no time ordering.

The arms (`sequential-lstm`, `sequential-transformer` in the unified runner
`src/fraud_pipeline.py`, implemented in `src/arms_sequential.py`): both read
the SAME per-customer transaction sequences — each customer's transactions
sorted by timestamp, every step carrying the 9 factor categoricals (type, ccy,
customer/counterparty country, customer_type, weekday, month, quarter, hour)
embedded plus amount_eur log-scaled — and differ only in the encoder (1-layer
LSTM, hidden 64 vs a 2-layer Transformer, d_model 64). NO label-derived
feature anywhere in a sequence — `fraud_flag` never enters the step tensors.

**Past-only scheme** (the leakage bar, one line per arm): the LSTM scores
transaction i from its own step embedding (the query) concatenated with the
recurrent state AFTER step i-1 — strictly earlier steps as context, step 0 of
a sequence getting a zero context state; the Transformer uses causal
self-attention keeping the diagonal (j <= i) — step i's own attributes enter
as its query (and its single self-value), every other attended step is
strictly earlier, and no future step ever contributes.

**Inference context, stated explicitly:** the customer-disjoint splits mean
the model NEVER trains on a test customer; within a test customer's sequence,
earlier test transactions serve as inference-time context only (no fitting, no
labels) — `predict_proba()` builds sequences from the rows it receives alone,
so the context is the customer's own earlier test transactions.

**Imbalance handling (honest disclosure):** `pos_weight = n_negative /
n_positive` over the training steps inside `BCEWithLogitsLoss` — the
loss-weight equivalent of the `scale_pos_weight` every tree arm bakes in.
Nothing beyond it: no oversampling, no resampling; the frozen-threshold tuning
on the runner's validation carve stays the threshold-side compensator.

Protocol (nb7-nb10 discipline, driven through the runner API — never
re-implemented here): one split per axis, stratified validation carve from
TRAIN only, threshold frozen on the carve, one-shot frozen-threshold test
evaluation with percentile-bootstrap AUC intervals. Both arms ship
`supports_cv=False` ([no-cv]); the early-stopping split is cut INSIDE fit() at
the CUSTOMER level (seed 42, stratified on has-fraud) — a row-level cut would
fragment per-customer sequences and leak a fit row into a val context. Every
results table below carries test positives and the chance level (the test
positive rate a random ranking lands at) — 91 frauds total make one unreadable
without the other.

In [1]:
# Runtime provenance - executed in the phase worktree venv built from the
# pinned requirements (kernel python3). Printed so the committed, executed
# notebook self-documents the exact runtime the numbers were produced on.
import platform
import sys

import numpy
import pandas
import sklearn
import torch

print('python:', sys.version.split()[0], '| platform:', platform.platform())
print('kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)')
for _mod in (pandas, numpy, sklearn, torch):
    print('%s: %s' % (_mod.__name__, _mod.__version__))
print('torch cuda available:', torch.cuda.is_available())
print('cuda device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu only')

python: 3.11.9 | platform: Windows-10-10.0.26200-SP0
kernel: python3 (nbclient + WindowsSelectorEventLoopPolicy)
pandas: 2.3.3
numpy: 2.4.6
sklearn: 1.7.2
torch: 2.11.0+cu128


torch cuda available: True
cuda device: NVIDIA GeForce RTX 5070 Ti Laptop GPU


## The arms through the runner — all three axes

`run_arm_on_axis` owns the whole protocol per axis: axis split → validation
carve → model → fit (early stopping on the arm's internal customer-level
carve) → threshold frozen on the runner's carve → test metrics + 1000-sample
percentile-bootstrap AUC CIs. `cv=False` here and both arms ship
`supports_cv=False` ([no-cv]): per-fold GPU refits would each re-pack
sequences, re-carve and re-run early stopping for little added evidence at
this scale. The axes, in registry order:

- **random-grouped (PRIMARY)** — seeded random customer assignment,
  customer-disjoint, no time ordering;
- **grouped (stress)** — test = latest-seen customers;
- **chronological (stress)** — test strictly later than train.

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

from arms_sequential import build_sequence_frame, fit_params, make_sequential
from fraud_pipeline import (
    AXES,
    DEFAULT_RESULTS_PATH,
    axis_split,
    best_f1_threshold,
    load_enriched,
    load_results,
    print_comparison_table,
    rich_test_metrics,
    run_arm_on_axis,
)

# Canonical enriched frame + labels from the unified runner (the nb7/nb8 data
# section, shared by every arm — loaded once, used by everything below).
enriched, y = load_enriched()
print(
    'fraud txns: %d of %d (%.2f%%) across %d unique customers'
    % (int(y.sum()), len(y), 100 * y.mean(), enriched['customer'].nunique())
)

fraud txns: 91 of 5302 (1.72%) across 100 unique customers


C:\Users\mateu\Documents\GitHub\la-wt\Fraud-Prediction\foc-175-f3\src\funs.py:197: UserWarning: The argument 'infer_datetime_format' is deprecated and will be removed in a future version. A strict version of it is now the default, see https://pandas.pydata.org/pdeps/0004-consistent-to-datetime-parsing.html. You can safely remove this argument.
  trxns_data["timestamp"] = pd.to_datetime(


## What the sequences look like

One sequence per customer, steps in timestamp order: 5302 transactions across
100 customers. The stats below are the context budget the models actually
see — how many strictly-earlier same-customer steps a transaction can read
(the first transaction of every sequence has none).

In [3]:
lengths = enriched.groupby('customer').size()
frauds_per_customer = enriched.assign(fraud=y).groupby('customer')['fraud'].sum()
past_steps = enriched.groupby('customer').cumcount()  # earlier same-customer steps per row

print('sequences: %d customers, %d steps total' % (len(lengths), int(lengths.sum())))
print(lengths.describe().round(1).to_string())
print(
    '\nfrauds per sequence: %d of %d customers carry all %d frauds, %d carry none'
    % (
        int((frauds_per_customer > 0).sum()), len(frauds_per_customer),
        int(frauds_per_customer.sum()), int((frauds_per_customer == 0).sum()),
    )
)
print(
    'past-only context per transaction: median %d, mean %.1f, max %d strictly-earlier '
    'same-customer steps; %d transactions (sequence starts) have zero context'
    % (
        int(past_steps.median()), float(past_steps.mean()), int(past_steps.max()),
        int((past_steps == 0).sum()),
    )
)

sequences: 100 customers, 5302 steps total
count    100.0
mean      53.0
std       18.6
min       19.0
25%       36.8
50%       52.0
75%       67.2
max       94.0

frauds per sequence: 25 of 100 customers carry all 91 frauds, 75 carry none
past-only context per transaction: median 26, mean 29.2, max 93 strictly-earlier same-customer steps; 100 transactions (sequence starts) have zero context


In [4]:
rows = []
for arm in ('sequential-lstm', 'sequential-transformer'):
    for axis in AXES:  # registry order: random-grouped (PRIMARY), grouped, chronological
        rows.append(run_arm_on_axis(arm, axis, enriched, y, cv=False))
print_comparison_table(rows, title='sequential arms — test metrics per axis (frozen threshold)')


== sequential arms — test metrics per axis (frozen threshold) ==
          axis                    arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high     f1  recall_at_precision  frozen_threshold  n_features
random-grouped        sequential-lstm     ok              13        977        0.0133  0.0167         0.0094          0.0359   0.5725          0.4470           0.6957 0.0000                  0.0            0.5967          12
random-grouped sequential-transformer     ok              13        977        0.0133  0.0158         0.0081          0.0403   0.4901          0.3006           0.6457 0.0260                  0.0            0.5103          12
       grouped        sequential-lstm     ok              11        831        0.0132  0.0353         0.0070          0.1934   0.4741          0.2894           0.6736 0.0274                  0.0            0.5935          12
       grouped sequential-transfor

## Sequential arms vs xgb-client — primary axis only

The F3 family question, head to head: do the sequence encoders beat the best
per-transaction arm (xgb-client, nb7 arm b) under the identical protocol on
the identical split? The xgb-client row comes from the accumulated results
file (`results/fraud_pipeline_results.jsonl`, latest row per axis x arm — the
same superseding convention the runner's table uses; if the row is missing the
cell re-runs the arm through the pipeline instead of guessing). All three rows
carry their test positive count, chance level and bootstrap CIs — with 13 test
positives the honest benchmark is distance-from-chance plus interval overlap,
not the point estimate alone.

In [5]:
def latest_ok(axis, arm):
    # Latest ok row per (axis, arm) from the accumulated JSONL.
    matches = [
        r
        for r in load_results(DEFAULT_RESULTS_PATH)
        if r.get('axis') == axis and r.get('arm') == arm and r.get('status') == 'ok'
    ]
    return matches[-1] if matches else None


lstm_row = next(r for r in rows if r['arm'] == 'sequential-lstm' and r['axis'] == 'random-grouped')
transformer_row = next(
    r for r in rows if r['arm'] == 'sequential-transformer' and r['axis'] == 'random-grouped'
)
xgb_row = latest_ok('random-grouped', 'xgb-client')
if xgb_row is None:  # results file unavailable/stale — re-run through the pipeline
    xgb_row = run_arm_on_axis('xgb-client', 'random-grouped', enriched, y, cv=False)

print_comparison_table(
    [lstm_row, transformer_row, xgb_row],
    title='sequential arms vs xgb-client — random-grouped (PRIMARY axis)',
)


def covers_chance(row):
    return row['pr_auc_ci_low'] <= row['chance_level'] <= row['pr_auc_ci_high']


for row in (lstm_row, transformer_row, xgb_row):
    print(
        '%-24s PR-AUC %.4f [CI %.4f, %.4f] vs chance %.4f (delta %+.4f) -> %s'
        % (
            row['arm'], row['pr_auc'], row['pr_auc_ci_low'], row['pr_auc_ci_high'],
            row['chance_level'], row['pr_auc'] - row['chance_level'],
            'indistinguishable from a random ranking' if covers_chance(row)
            else 'separates from chance (read the interval)',
        )
    )


== sequential arms vs xgb-client — random-grouped (PRIMARY axis) ==
          axis                    arm status  test_positives  test_rows  chance_level  pr_auc  pr_auc_ci_low  pr_auc_ci_high  roc_auc  roc_auc_ci_low  roc_auc_ci_high    f1  recall_at_precision  frozen_threshold  cv_pr_auc_mean  cv_pr_auc_std  n_features
random-grouped             xgb-client     ok              13        977        0.0133  0.0144         0.0079          0.0300   0.4844          0.3334           0.6425 0.000                  0.0            0.8776          0.6812         0.0768         116
random-grouped        sequential-lstm     ok              13        977        0.0133  0.0167         0.0094          0.0359   0.5725          0.4470           0.6957 0.000                  0.0            0.5967             NaN            NaN          12
random-grouped sequential-transformer     ok              13        977        0.0133  0.0158         0.0081          0.0403   0.4901          0.3006           0.6457

## Protocol refit, early-stopping evidence, determinism

`run_arm_on_axis` fits internally and discards the model, so both arms are
rebuilt here under the exact primary-axis protocol (same split, same carve,
same seeds) for the diagnostics cell. Three reads off these fits:

- **Early-stopping carve.** The arm cuts its own carve at the CUSTOMER level
  (stratified on has-fraud, seed 42) — customer-disjoint by construction, so
  unlike nb10's customer-mixed carve it is itself a small held-out-customer
  readout inside the fitting distribution. Its PR-AUC next to the test-axis
  numbers says whether ANY sequence signal generalizes across customers.
- **Sanity.** Each refit's frozen-threshold test metrics must mirror the
  runner's random-grouped row from the first table.
- **Determinism.** Two additional fits per arm under the same seeds; the max
  absolute probability deviation is printed and reported honestly —
  `cudnn.deterministic` is an attempt, not a guarantee, and the committed
  outputs state what was actually observed.

In [6]:
X_raw = build_sequence_frame(enriched)
train_idx, test_idx = axis_split('random-grouped', enriched, y)
X_tr, X_te = X_raw.loc[train_idx], X_raw.loc[test_idx]
y_tr, y_te = y.loc[train_idx], y.loc[test_idx]
X_fit, X_val, y_fit, y_val = train_test_split(
    X_tr, y_tr, test_size=0.25, random_state=42, stratify=y_tr
)
runner_rows = {r['arm']: r for r in rows if r['axis'] == 'random-grouped'}

for arm_name, kind in (('sequential-lstm', 'lstm'), ('sequential-transformer', 'transformer')):
    print('== %s ==' % arm_name)
    fit_model = make_sequential(kind, y_fit).fit(X_fit, y_fit)
    carve = fit_model.carve_
    print(
        'internal carve (customer-level): %d fit customers (%d steps, %d positives) / '
        '%d val customers (%d steps, %d positives) | pos_weight %.2f | device %s'
        % (
            carve['fit_customers'], carve['fit_steps'], carve['fit_positives'],
            carve['val_customers'], carve['val_steps'], carve['val_positives'],
            carve['pos_weight'], fit_model.device_name_,
        )
    )
    print(
        'early stopping: best carve PR-AUC %.4f at epoch %d of %d run (patience %d)'
        % (
            fit_model.best_val_pr_auc_, fit_model.best_epoch_, fit_model.epochs_run_,
            fit_params()['patience'],
        )
    )

    # Sanity: identical protocol, identical seeds -> must mirror the runner row.
    threshold = best_f1_threshold(y_val, fit_model.predict_proba(X_val)[:, 1])
    proba_a = fit_model.predict_proba(X_te)[:, 1]
    refit_metrics = rich_test_metrics(y_te, proba_a, threshold)
    print(
        'refit sanity: test PR-AUC %.4f vs runner row %.4f | F1 %.4f | threshold %.4f'
        % (
            refit_metrics['pr_auc'], runner_rows[arm_name]['pr_auc'],
            refit_metrics['f1'], threshold,
        )
    )

    # Determinism: two more fits, same seeds, same data -> measure the deviation.
    proba_b = make_sequential(kind, y_fit).fit(X_fit, y_fit).predict_proba(X_te)[:, 1]
    proba_c = make_sequential(kind, y_fit).fit(X_fit, y_fit).predict_proba(X_te)[:, 1]
    max_dev = float(max(np.max(np.abs(proba_a - proba_b)), np.max(np.abs(proba_b - proba_c))))
    print(
        'within-kernel refit determinism (3 fits, same seeds): max |delta proba| = %.3e -> %s'
        % (max_dev, 'bit-identical predictions' if max_dev == 0.0 else 'GPU nondeterminism observed')
    )

== sequential-lstm ==


internal carve (customer-level): 60 fit customers (2447 steps, 46 positives) / 20 val customers (796 steps, 12 positives) | pos_weight 52.20 | device cuda
early stopping: best carve PR-AUC 0.0229 at epoch 5 of 13 run (patience 8)
refit sanity: test PR-AUC 0.0167 vs runner row 0.0167 | F1 0.0000 | threshold 0.5967


within-kernel refit determinism (3 fits, same seeds): max |delta proba| = 0.000e+00 -> bit-identical predictions
== sequential-transformer ==


internal carve (customer-level): 60 fit customers (2447 steps, 46 positives) / 20 val customers (796 steps, 12 positives) | pos_weight 52.20 | device cuda
early stopping: best carve PR-AUC 0.0221 at epoch 3 of 11 run (patience 8)
refit sanity: test PR-AUC 0.0158 vs runner row 0.0158 | F1 0.0260 | threshold 0.5103


within-kernel refit determinism (3 fits, same seeds): max |delta proba| = 0.000e+00 -> bit-identical predictions


### Interpretation (read after the tables — null results are findings)

- **Noise budget first.** 91 frauds total; the three splits put 13
  (random-grouped), 11 (grouped) and 24 (chronological) test positives in play
  — printed on every table row next to its chance level. At that size a single
  swapped fraud moves test PR-AUC by hundredths and the bootstrap CIs span a
  wide band around every point estimate: deltas under ~0.05 PR-AUC between
  arms are noise, not signal.
- **Read every number against its chance level.** Chance PR-AUC equals the
  test positive rate: 0.0133 on random-grouped, 0.0132 on grouped, 0.0226 on
  chronological. The sequential rows (first table) and the xgb-client
  comparison (second table) must be judged by their distance from those
  baselines AND by their printed CIs — an interval that still covers the
  chance level means the model is indistinguishable from a random ranking on
  that split, whatever the point estimate suggests. A null result here is a
  finding, not a failure: it says per-customer recurrence over ~50-step
  sequences extracts nothing the per-transaction features did not, at this
  data size — 91 frauds across 100 customers give the encoders ~19 fitting
  customers' worth of fraud signal to learn "unusual for THIS customer" from.
- **Axis structure.** random-grouped is PRIMARY (customer-disjoint but
  temporally unbiased); grouped and chronological are stress tests that also
  remove recency overlap. nb7-nb10 showed the client-feature lift is
  customer-identity signal that collapses once test customers are disjoint
  from train; the sequence arms consume no client attributes at all — their
  only customer-specific signal is the behavior inside the sequence — so a
  collapse here is cleaner evidence about the sequence hypothesis itself.
- **The internal carve is customer-disjoint.** Unlike nb10's customer-mixed
  early-stopping carve, this arm's carve keeps whole customers out of
  training, so its val PR-AUC (printed above) is the closest analogue to the
  nb9 CV summary: a high carve value next to chance-level customer-disjoint
  test metrics would mean the encoders fit sequence structure that does not
  transfer to UNSEEN customers — the memorization signature without identity
  features to memorize. Read the gap as generalization evidence, not as a bug
  in the protocol.
- **Past-only discipline, restated.** Transaction i is scored from its own
  attributes (the query) plus strictly earlier same-customer steps (context):
  the LSTM reads the recurrent state after step i-1, the Transformer attends
  under a causal mask keeping the diagonal (j <= i). No label ever enters a
  sequence; on the customer-disjoint axes the test context is the customer's
  own earlier test transactions only (no fitting, no labels) — stated in the
  title cell and realized by `predict_proba()` packing the rows it receives.
- **Imbalance flag.** Both arms handle the 1.72% positive rate ONLY through
  `pos_weight` in `BCEWithLogitsLoss` (the loss-weight equivalent of the tree
  arms' `scale_pos_weight`) — disclosed, no oversampling anywhere. The
  reported F1 / recall-at-precision rest on the frozen-threshold validation
  tuning.
- **Determinism.** Seeds 42 everywhere (splits, carves, torch, numpy, the
  fit-local permutation RNG), `cudnn.deterministic=True` with
  `cudnn.benchmark=False` — an attempt, not a guarantee. Observed verdict,
  reported rather than smoothed over: within each arm three identical fits
  inside one kernel produced the deviation printed above, and two full CLI
  invocations per arm (`fraud_pipeline.py --run-arm ... --axis random-grouped`)
  reproduced identical test metrics to the last digit — only wall-clock
  differed — so the committed numbers are reproducible on this stack.
- **The direction, not the score, is the finding.** Per-customer sequence
  context is the genuinely valuable NN direction on this problem: a production
  stream carries full customer histories, while this table gives every model
  5302 transactions, 100 customers and 91 frauds — far too little for a
  sequence encoder to learn a transferable notion of "unusual for THIS
  customer". The honest conclusion is not "sequence models fail" but "at this
  scale nothing separates them from a random ranking"; worth re-testing when
  more data arrives.

## Summary

- `src/arms_sequential.py` implements the two F3 sequence arms: a 1-layer LSTM
  (hidden 64) and a 2-layer Transformer encoder (d_model 64), both reading
  per-customer transaction sequences built from the 9 factor categoricals
  (embedded) + log-scaled amount_eur — no label-derived feature anywhere.
  Past-only scoring is structural: shifted recurrent state for the LSTM,
  causal attention keeping the diagonal for the Transformer.
- Both arms are registered in the unified runner (`src/fraud_pipeline.py`) —
  replacing the `sequential` placeholder — with `supports_cv=False`, and are
  exercised end-to-end here through `run_arm_on_axis` on all three axes; the
  primary-axis rows are benchmarked against xgb-client from the accumulated
  results file.
- Imbalance is handled only by the disclosed `pos_weight` in
  `BCEWithLogitsLoss`; the frozen-threshold protocol is unchanged (nb7-nb11).
- The verdicts are read from the tables above against their chance levels and
  bootstrap CIs; the committed CLI runs append the same protocol's numbers to
  `results/fraud_pipeline_results.jsonl`.
- Closing position: the per-customer sequence direction is the genuinely
  valuable NN direction here — null at this data size, worth re-testing when
  more data arrives.